In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/datasets/amulyajm1/world-happiness/world_happiness_2023.csv


In [2]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/kaggle/input/datasets/amulyajm1/world-happiness/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

In [4]:
# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


In [5]:
import plotly.graph_objects as go

# Pull out the key facts for the insight title
top_region = region_avg.iloc[-1]      # highest (last row, since sorted ascending)
bottom_region = region_avg.iloc[0]    # lowest
gap = top_region['Happiness_Score'] - bottom_region['Happiness_Score']

# Design choice: highlight the top region, mute the rest
colors = ['#C9CED6'] * len(region_avg)
colors[-1] = '#1F77B4'

fig = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text=(f"<b>{top_region['Region']} is the happiest region, "
              f"scoring {gap:.1f} points above {bottom_region['Region']}</b>"
              "<br><sup>Average happiness score by region (0–10 scale)</sup>"),
        x=0.0, xanchor='left'
    ),
    xaxis=dict(
        range=[0, region_avg['Happiness_Score'].max() * 1.15],  # zero baseline
        title='Average happiness score',
        showgrid=True, gridcolor='#EEEEEE', zeroline=False
    ),
    yaxis=dict(title='', automargin=True),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#333333'),
    margin=dict(l=20, r=40, t=100, b=50),
    height=500,
    showlegend=False
)

# Annotation: global average reference
global_avg = df['Happiness_Score'].mean()
fig.add_vline(x=global_avg, line_dash='dash', line_color='#E4572E', line_width=1.5)
fig.add_annotation(
    x=global_avg, y=1.02, yref='paper',
    text=f"Global avg: {global_avg:.2f}",
    showarrow=False, font=dict(color='#E4572E', size=12),
    xanchor='left'
)

fig.show()

In [6]:
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

Global average: 5.81


In [7]:
import plotly.graph_objects as go

# Colours: teal for top group, coral for bottom group
color_map = {'Top 8': '#1B7F79', 'Bottom 8': '#D1495B'}
bar_colors = combined['Group'].map(color_map)

# Gap facts for the insight title
top_avg = top8['Happiness_Score'].mean()
bottom_avg = bottom8['Happiness_Score'].mean()
ratio = top_avg / bottom_avg

fig = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker_color=bar_colors,
    text=combined['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text=(f"<b>The happiest countries score {ratio:.1f}x higher than the least happy</b>"
              f"<br><sup>Top 8 average: {top_avg:.2f} vs. Bottom 8 average: {bottom_avg:.2f} "
              f"(happiness score, 0–10 scale)</sup>"),
        x=0.0, xanchor='left'
    ),
    xaxis=dict(
        range=[0, combined['Happiness_Score'].max() * 1.15],  # zero baseline
        title='Happiness score',
        showgrid=True, gridcolor='#EEEEEE', zeroline=False
    ),
    yaxis=dict(title='', automargin=True),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#333333'),
    margin=dict(l=20, r=60, t=110, b=50),
    height=700,
    showlegend=False
)

# Visual separator: dashed line between the two groups (between bar 8 and bar 9)
fig.add_hline(y=7.5, line_dash='dash', line_color='#888888', line_width=1.5)

# Global average reference line
fig.add_vline(x=global_avg, line_dash='dot', line_color='#555555', line_width=1.2)
fig.add_annotation(
    x=global_avg, y=1.02, yref='paper',
    text=f"Global avg: {global_avg:.2f}",
    showarrow=False, xanchor='left', font=dict(color='#555555', size=12)
)

# Group labels
fig.add_annotation(
    x=0.98, xref='paper', y=15, yref='y',
    text="<b>Top 8</b>", showarrow=False, xanchor='right',
    font=dict(color='#1B7F79', size=15)
)
fig.add_annotation(
    x=0.98, xref='paper', y=0, yref='y',
    text="<b>Bottom 8</b>", showarrow=False, xanchor='right',
    font=dict(color='#D1495B', size=15)
)

# Gap annotation placed on the separator line
fig.add_annotation(
    x=0.98, xref='paper', y=7.5, yref='y',
    text=f"<b>Gap: {top_avg - bottom_avg:.2f} points</b>",
    showarrow=False, xanchor='right', yshift=12,
    font=dict(color='#333333', size=13)
)

fig.show()